In [1]:
import geopandas as gpd
from sqlalchemy import create_engine, text
import os
import numpy as np

In [ ]:
def load_vector_layer(db_name, user, password, host, port, table_name, schema='public', geom_col='geom'):
    """
    Connects to a PostGIS-enabled PostgreSQL database and loads a vector layer as a GeoDataFrame.
    
    Parameters:
    - db_name (str): Name of the PostgreSQL database.
    - user (str): Database username.
    - password (str): Database password.
    - host (str): Host address (e.g., 'localhost' or IP).
    - port (int): Port number (e.g., 5432).
    - table_name (str): Name of the table (vector layer) to load.
    - schema (str): Optional. Database schema containing the table (default is 'public').

    Returns:
    - gpd.GeoDataFrame: A GeoDataFrame containing the vector layer.
    """
    try:
        # Use pg8000 (pure Python driver)
        conn_str = f"postgresql+pg8000://{user}:{password}@{host}:{port}/{db_name}"
        engine = create_engine(conn_str)

        sql = text(f'SELECT * FROM "{schema}"."{table_name}"')

        # Open a connection explicitly (SQLAlchemy 2.x requirement)
        with engine.connect() as conn:
            gdf = gpd.read_postgis(sql, conn, geom_col=geom_col)
        
        print(f"Successfully loaded {table_name} ({len(gdf)} features)")
        return gdf

    except Exception as e:
        print(f"Error loading vector layer: {e}")
        return None
    
def split_by_grid(gdf_input, gdf_grid, index_col='tile_index'):
    # Optional but VERY important for speed
    # (Shapely 2 / PyGEOS backend)
    gdf = gdf_input.copy()
    grid = gdf_grid.copy()

    gdf_geometry_name = gdf.geometry.name
    grid_geometry_name = grid.geometry.name

    # Make geometries valid (prevents topology errors & slowdowns)
    gdf[gdf_geometry_name] = gdf.make_valid()
    grid[grid_geometry_name] = grid.make_valid()

    # Keep only needed columns
    # be careful with the columns
    grid = grid[[index_col, "ISO_SOV1", grid_geometry_name]]   # cell_id = your grid unique ID

    # ---- SPLIT coastline by grid (spatial-index accelerated) ----
    split = gpd.overlay(
        gdf,
        grid,
        how="intersection",
        keep_geom_type=True
    )

    # ---- DISSOLVE per grid cell ----
    # result = split.dissolve(
    #     by=index_col,
    #     as_index=False
    # )

    return split # result



def publish_vector_layer(gdf, db_name, user, password, host, port,
                         table_name, schema='public', geom_col='geom'):
    """
    Publishes a GeoDataFrame to a PostGIS-enabled PostgreSQL database.

    Parameters:
    - gdf (gpd.GeoDataFrame): GeoDataFrame to upload.
    - db_name (str): Database name.
    - user (str): Username.
    - password (str): Password.
    - host (str): Host address.
    - port (int): Port number.
    - table_name (str): Target table name.
    - schema (str): Schema (default 'public').
    - geom_col (str): Geometry column name (default 'geom').

    Returns:
    - None
    """

    try:
        # Connection string
        conn_str = f"postgresql+psycopg://{user}:{password}@{host}:{port}/{db_name}"
        engine = create_engine(conn_str)

        # Ensure geometry column name matches
        if gdf.geometry.name != geom_col:
            gdf = gdf.rename_geometry(geom_col)

        # Write to PostGIS
        gdf.to_postgis(
            name=table_name,
            con=engine,
            schema=schema,
            if_exists='fail',
            index=True
        )

        print(f"Successfully published {table_name} ({len(gdf)} features)")

    except Exception as e:
        print(f"Error publishing vector layer: {e}")

In [7]:
gdf_1d_grid = load_vector_layer(
    db_name='geoserver',
    user='geoserver',
    password='geoserver',
    host='192.168.250.233',
    port=5555,
    table_name='global_klab_1d_tiles',
    schema='klab_grids'
)

# gdf_3d_grid = load_vector_layer(
#     db_name='geoserver',
#     user='geoserver',
#     password='geoserver',
#     host='192.168.250.233',
#     port=5555,
#     table_name='global_klab_3d_tiles',
#     schema='klab_grids'
# )


osm_coast_gdf = load_vector_layer(
    db_name='geoserver',
    user='geoserver',
    password='geoserver',
    host='192.168.250.100',
    port=5555,
    table_name='global_osm_coastline_6km_buffer_grid_split_4326',
    schema='public'
)

# un_gdf = load_vector_layer(
#     db_name='geoserver',
#     user='geoserver',
#     password='geoserver',
#     host='192.168.250.100',
#     port=5555,
#     table_name='administrative-level-0-un-split',
#     schema='public'
# )

Successfully loaded global_klab_1d_tiles (50760 features)
Successfully loaded global_osm_coastline_6km_buffer_grid_split_4326 (6540 features)


In [ ]:
gdf_grid = gpd.read_file(r"C:\Users\ruben.crespo\Downloads\0_marine\EEZ_land_union_v4_202410.shp")

In [ ]:
gdf_grid.head()

,UNION,MRGID_EEZ,TERRITORY1,MRGID_TER1,ISO_TER1,UN_TER1,SOVEREIGN1,MRGID_SOV1,ISO_SOV1,UN_SOV1,...,UN_TER3,SOVEREIGN3,MRGID_SOV3,ISO_SOV3,UN_SOV3,POL_TYPE,Y_1,x_1,AREA_KM2,geometry
0,Jordan,8491.0,Jordan,2120.0,JOR,400.0,Jordan,2120.0,JOR,400.0,...,NaN,None,NaN,None,NaN,Union EEZ and country,31.24507,36.78660,89444,"POLYGON ((38.98446 32.47799, 39.04382 32.30118..."
1,Burundi,NaN,Burundi,2172.0,BDI,108.0,Burundi,2172.0,BDI,108.0,...,NaN,None,NaN,None,NaN,Landlocked country,-3.36600,29.89129,26833,"POLYGON ((30.42148 -2.32618, 30.42277 -2.32734..."
2,Uruguay,8467.0,Uruguay,2216.0,URY,858.0,Uruguay,2216.0,URY,858.0,...,NaN,None,NaN,None,NaN,Union EEZ and country,-34.15741,-54.78946,320453,"POLYGON ((-52.99951 -37.70660, -53.12598 -37.6..."
3,Latvia,5683.0,Latvia,2132.0,LVA,428.0,Latvia,2132.0,LVA,428.0,...,NaN,None,NaN,None,NaN,Union EEZ and country,56.93080,23.83336,92873,"POLYGON ((26.58270 55.67484, 26.57660 55.67558..."
4,Kuril Islands,48950.0,Kuril Islands,9011.0,None,NaN,Japan,2121.0,JPN,392.0,...,NaN,None,NaN,None,NaN,Overlapping claim,43.90538,147.94430,218735,"POLYGON ((151.03846 42.40393, 151.02995 42.391..."


In [ ]:
gdf_input = split_by_grid(osm_coast_gdf, gdf_grid, index_col='TERRITORY1')

In [9]:
gdf_input.head()

,gid,tile_index_1d,tile_index_3d,TERRITORY1,ISO_SOV1,geometry
0,1,11992,1358,Chile,CHL,"POLYGON ((-68.74533 -56.42005, -68.74500 -56.4..."
1,2,11993,1358,Chile,CHL,"POLYGON ((-67.24486 -56.00141, -67.24706 -56.0..."
2,5,12349,1357,Chile,CHL,"POLYGON ((-71.00067 -55.12140, -71.00101 -55.1..."
3,6,12350,1357,Chile,CHL,"MULTIPOLYGON (((-70.00311 -55.44557, -70.00554..."
4,7,12351,1357,Chile,CHL,"MULTIPOLYGON (((-69.00990 -55.50102, -69.01091..."


In [10]:
gdf_input.to_file("global_osm_coastline_6km_buffer_tada.shp", driver="ESRI Shapefile")

C:\Users\ruben.crespo\AppData\Local\Temp\ipykernel_8012\3968955002.py:1: UserWarning: Column names longer than 10 characters will be truncated when saved to ESRI Shapefile.
  gdf_input.to_file("global_osm_coastline_6km_buffer_tada.shp", driver="ESRI Shapefile")


In [ ]:
publish_vector_layer(gdf_input, 
                     db_name="geoserver", 
                     user= "geoserver", 
                     password= "geoserver", 
                     host='192.168.250.100', 
                     port=5555,
                     table_name="administrative_units_un_gadm_level0_1d_tile_index_v2", 
                     schema='public', 
                     geom_col='geom')

Error publishing vector layer: 'Cursor' object has no attribute 'copy_expert'
